# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinukondablessena/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [33]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir("."))


Current working directory:
/content/flyrank-ml-internship

Files/folders here:
['README.md', 'work', 'notebooks', 'requirements.txt', '.github', 'skills', 'LICENSE', 'outputs', 'docs', 'submission', 'SETUP.md', 'scripts', '.git', 'data', 'DATA_USE.md', 'AGENTS.md', '.gitignore', 'CLAUDE.md', 'GUIDE.md']


In [34]:
!git clone https://github.com/vinukondablessena/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 139 (delta 49), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.86 MiB | 4.88 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [35]:
import os

print(os.listdir("/content/flyrank-ml-internship"))

['README.md', 'work', 'notebooks', 'requirements.txt', '.github', 'skills', 'LICENSE', 'outputs', 'docs', 'submission', 'SETUP.md', 'scripts', 'flyrank-ml-internship', '.git', 'data', 'DATA_USE.md', 'AGENTS.md', '.gitignore', 'CLAUDE.md', 'GUIDE.md']


In [36]:
import os

data_path = "/content/flyrank-ml-internship/data"

print("Data folder contents:")
print(os.listdir(data_path))

Data folder contents:
['raw', 'processed']


In [37]:
import os

raw_path = "/content/flyrank-ml-internship/data/raw"

print("Raw folder contents:")
print(os.listdir(raw_path))

Raw folder contents:
['content_refresh_anonymized.csv']


In [38]:
print("All columns:")
for i, col in enumerate(df.columns, start=1):
    print(i, repr(col))

All columns:
1 'content_id'
2 'client_id'
3 'search_volume'
4 'competition'
5 'competition_level'
6 'cpc'
7 'content_type'
8 'main_intent'
9 'word_count'
10 'char_count'
11 'provider_used'
12 'model_used'
13 'impressions_90d'
14 'clicks_90d'
15 'pageviews_90d'
16 'sessions_90d'
17 'users_90d'
18 'engaged_sessions_90d'
19 'ai_sessions_90d'
20 'scroll_events_90d'
21 'days_with_impressions'
22 'days_with_sessions'
23 'impressions_last_30d'
24 'clicks_last_30d'
25 'sessions_last_30d'
26 'impressions_prev_30d'
27 'clicks_prev_30d'
28 'sessions_prev_30d'
29 'content_age_days'
30 'age_tier'
31 'age_tier_order'
32 'days_since_last_update'
33 'freshness_tier'
34 'word_count_tier'
35 'char_count_tier'
36 'ctr'
37 'avg_position'
38 'engagement_rate'
39 'scroll_rate'
40 'ai_traffic_pct'
41 'impression_tier'
42 'position_tier'
43 'trend_direction'
44 'trend_pct'
45 'is_declining_label'
46 'log_impressions_90d'
47 'log_clicks_90d'
48 'log_sessions_90d'
49 'log_ai_sessions_90d'
50 'has_clicks'
51 'ha

In [39]:
import os

dictionary_path = "/content/flyrank-ml-internship/docs/data-dictionary.md"

print("File exists:", os.path.exists(dictionary_path))

with open(dictionary_path, "r", encoding="utf-8") as f:
    dictionary = f.read()

print(dictionary[:12000])

File exists: True
# Data Dictionary — `content_refresh_anonymized.csv`

One row per content item (page): **30,000 rows × 44 columns**, covering **32 pseudonymized
clients**. All metrics are aggregated over a trailing 90-day window ending at export time.
Keep this file open while you work.

## Read this first — the three rules that prevent 90% of mistakes

1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
2. **The label comes from `trend_direction`.** The pipeline defines
   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
   must **never** be model features — that's the leakage notebook 02 demonstrates.
3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
   joins and grouped train/test splits, never as features.

## Identifiers

| Column | Type | Meaning | Notes |
|---|---|---|---|
| `content_i

In [40]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if "refresh_feature" in file.lower():
            print(os.path.join(root, file))

/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [41]:
script_path = "/content/flyrank-ml-internship/scripts/01_prepare_features.py"

with open(script_path, "r", encoding="utf-8") as f:
    script = f.read()

print(script)

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_utils import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
    PROCESSED_DIR,
    RAW_PATH,
    display_path,
    ensure_dirs,
    to_bool_series,
    write_json,
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prepare FlyRank refresh feature vector.")
    parser.add_argument("--input", default=str(RAW_PATH), help="Raw anonymized CSV export.")
    parser.add_argument(
        "--output",
        default=str(PROCESSED_DIR / "refresh_feature_vector.csv"),
        help="Prepared feature-vector CSV.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    ensure_dirs()

    input_path = Path(args.input)
    if not input_path.exists():
        raise FileNotFoundError(
            f"Raw input not found: {input

In [42]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [43]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [44]:
import pandas as pd
import os

PROCESSED_PATH = "/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"

print("File exists:", os.path.exists(PROCESSED_PATH))

df = pd.read_csv(PROCESSED_PATH)

print("Shape:", df.shape)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

print("\nFirst 10 columns:")
print(df.columns[:10].tolist())

File exists: True
Shape: (30000, 52)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667

First 10 columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


In [45]:
import sys

sys.path.insert(0, "/content/flyrank-ml-internship/scripts")

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

print("Numeric features:")
print(MODEL_NUMERIC_FEATURES)

print("\nNumber of numeric features:", len(MODEL_NUMERIC_FEATURES))

print("\nCategorical features:")
print(MODEL_CATEGORICAL_FEATURES)

print("\nNumber of categorical features:", len(MODEL_CATEGORICAL_FEATURES))

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Number of numeric features: 18

Categorical features:
['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Number of categorical features: 8


In [46]:
# Leakage and ID checks

LEAKAGE_COLUMNS = {
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "is_declining_label"
}

all_features = list(MODEL_NUMERIC_FEATURES) + list(MODEL_CATEGORICAL_FEATURES)

leakage_found = [col for col in all_features if col in LEAKAGE_COLUMNS]

print("Total model features:", len(all_features))
print("Leakage/ID columns found in feature lists:", leakage_found)

print("\nTarget column present in dataset:", "is_declining_label" in df.columns)
print("Client column present:", "client_id" in df.columns)

print("\nNumber of unique clients:", df["client_id"].nunique())

Total model features: 26
Leakage/ID columns found in feature lists: []

Target column present in dataset: True
Client column present: True

Number of unique clients: 32


In [47]:
from sklearn.model_selection import GroupShuffleSplit

TARGET = "is_declining_label"

X = df.drop(columns=[TARGET])
y = df[TARGET]
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nClients appearing in BOTH sets:")
print(set(train_df["client_id"]) & set(test_df["client_id"]))

print("\nTrain target rate:", train_df[TARGET].mean())
print("Test target rate:", test_df[TARGET].mean())

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Clients appearing in BOTH sets:
set()

Train target rate: 0.5501111717078492
Test target rate: 0.5109524582184002


We use Logistic Regression as the first learned model because this is a binary classification problem: predicting whether a content item is declining (`is_declining_label = 1`). Logistic Regression is a simple and interpretable baseline for a learned model, so it gives us a clear comparison against the Week-4 rule baseline without rewarding unnecessary complexity. We use the prepared feature set from the repository and exclude the label source (`trend_direction` and `trend_pct`) and IDs from the model features.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

We use a client-grouped 80/20 train/test split with a fixed random seed of 42. Entire clients are held out from the test set, so no client appears in both training and testing. This is more honest than a random row split because multiple content items belong to the same client and could otherwise create overly optimistic results. The model and Week-4 baseline will be evaluated on the same held-out test data and the same metric.

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Features from the official FlyRank feature lists
numeric_features = list(MODEL_NUMERIC_FEATURES)
categorical_features = list(MODEL_CATEGORICAL_FEATURES)

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[TARGET]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df[TARGET]

# Numeric preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Logistic Regression
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Model trained successfully.")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Model trained successfully.
Training rows: 23837
Test rows: 6163


In [49]:
from sklearn.metrics import precision_score, accuracy_score, roc_auc_score

# Predicted probability of the declining class
test_prob = model.predict_proba(X_test)[:, 1]

# Predicted class using the default 0.5 threshold
test_pred = (test_prob >= 0.5).astype(int)


def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    # Highest-risk items first
    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()


p20 = precision_at_k(y_test.values, test_prob, 20)
p50 = precision_at_k(y_test.values, test_prob, 50)

accuracy = accuracy_score(y_test, test_pred)
roc_auc = roc_auc_score(y_test, test_prob)

print("Logistic Regression results")
print("---------------------------")
print(f"Precision@20: {p20:.4f}")
print(f"Precision@50: {p50:.4f}")
print(f"Accuracy:     {accuracy:.4f}")
print(f"ROC-AUC:      {roc_auc:.4f}")

Logistic Regression results
---------------------------
Precision@20: 0.7000
Precision@50: 0.7200
Accuracy:     0.5832
ROC-AUC:      0.6158


In [50]:
import os

print("Searching for baseline_action_score.csv...")

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if "baseline" in file.lower() or "action_score" in file.lower():
            print(os.path.join(root, file))

Searching for baseline_action_score.csv...
/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ml-internship/scripts/02_baseline_score.py
/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/scripts/02_baseline_score.py
/content/flyrank-ml-internship/data/processed/baseline_metadata.json
/content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv


In [51]:
baseline_script = "/content/flyrank-ml-internship/scripts/02_baseline_score.py"

with open(baseline_script, "r", encoding="utf-8") as f:
    baseline_code = f.read()

print(baseline_code)

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_utils import PROCESSED_DIR, normalize, percentile_rank, write_json


FEATURE_PATH = PROCESSED_DIR / "refresh_feature_vector.csv"
OUTPUT_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Build deterministic refresh baseline score.")
    parser.add_argument("--input", default=str(FEATURE_PATH))
    parser.add_argument("--output", default=str(OUTPUT_PATH))
    return parser.parse_args()


def reason_codes(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and

In [52]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [53]:
!python scripts/02_baseline_score.py

Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


In [54]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if "baseline_action_score" in file.lower():
            print(os.path.join(root, file))

In [55]:
import os

print("Outputs folder:")
print(os.listdir("/content/flyrank-ml-internship/outputs"))

print("\nWork folder:")
print(os.listdir("/content/flyrank-ml-internship/work"))

print("\nWork/outputs folder:")
work_outputs = "/content/flyrank-ml-internship/work/outputs"

if os.path.exists(work_outputs):
    print(os.listdir(work_outputs))
else:
    print("work/outputs does not exist")

Outputs folder:
['model_report.md', 'charts', 'refresh_queue_sample.csv']

Work folder:
['capstone_report_template.md', 'README.md', 'notebooks']

Work/outputs folder:
work/outputs does not exist


In [56]:
import json

notebook_path = "/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    print(f"\n========== CELL {i} ({cell['cell_type']}) ==========")
    print("".join(cell["source"])[:3000])


========== CELL 0 (markdown) ==========
# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinukondablessena/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

========== CELL 1 (markdown) ==========
## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### My baseline rule

I will rank content pages for refresh review using two observable signals: content freshness and search visibility. Pages that have not been updated for a long time but still have meaningful impressions will receive a higher review score. The rule i

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [57]:
# Recreate the exact ML-07 baseline rule on the ML-08 test set

baseline_test = test_df.copy()

baseline_test["baseline_score"] = (
    (baseline_test["impressions_90d"] >= 500).astype(int) * 40
    + (
        (baseline_test["avg_position"] >= 1)
        & (baseline_test["avg_position"] <= 10)
    ).astype(int) * 30
    + (
        (baseline_test["avg_position"] > 10)
        & (baseline_test["avg_position"] <= 20)
    ).astype(int) * 20
    + (baseline_test["ctr"] < 0.5).astype(int) * 10
)

baseline_p20 = precision_at_k(
    baseline_test[TARGET].values,
    baseline_test["baseline_score"].values,
    20
)

baseline_p50 = precision_at_k(
    baseline_test[TARGET].values,
    baseline_test["baseline_score"].values,
    50
)

print("ML-07 Baseline results on ML-08 test set")
print("------------------------------------------")
print(f"Precision@20: {baseline_p20:.4f}")
print(f"Precision@50: {baseline_p50:.4f}")


ML-07 Baseline results on ML-08 test set
------------------------------------------
Precision@20: 0.6500
Precision@50: 0.7400


In [58]:
comparison = pd.DataFrame({
    "Method": [
        "ML-07 Rule Baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        p20
    ],
    "Precision@50": [
        baseline_p50,
        p50
    ]
})

print(comparison.to_string(index=False))

             Method  Precision@20  Precision@50
ML-07 Rule Baseline          0.65          0.74
Logistic Regression          0.70          0.72


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*                                               

  ### Error analysis and interpretation

The Logistic Regression model performs better than the ML-07 rule baseline at Precision@20 (0.70 vs 0.65), but the baseline performs better at Precision@50 (0.74 vs 0.72). This suggests that the learned model is useful for prioritizing a very small set of high-risk pages, while the transparent rule remains competitive for a larger review queue.

The model's errors are expected because declining performance can depend on factors that are not fully represented by the available features. Pages may be difficult to classify when their traffic is low, their search position is unstable, or their content characteristics do not clearly distinguish declining from non-declining pages.

Feature interpretation and three concrete incorrect predictions are inspected below. These examples are treated as observed model errors rather than evidence of causation.

In [59]:


feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values("abs_coefficient", ascending=False)

print("Top 10 features by absolute coefficient:")
display(feature_importance.head(10)[["feature", "coefficient"]])


Top 10 features by absolute coefficient:


,feature,coefficient
5,num__log_impressions_90d,1.425272
51,cat__position_tier_top_3,-0.802077
45,cat__impression_tier_low,0.725328
43,cat__impression_tier_excellent,-0.622277
38,cat__word_count_tier_1000-2000,0.615490
6,num__log_clicks_90d,-0.571279
29,cat__main_intent_unknown,0.533893
3,num__word_count,0.508241
21,cat__competition_level_unknown,-0.482846
24,cat__content_type_keyword article,0.451658


### Feature interpretation

The strongest observed model signals are `log_impressions_90d`, `position_tier_top_3`, and `impression_tier_low`. The positive coefficient for `log_impressions_90d` means higher historical impressions are associated with a higher predicted probability of decline in this fitted model. The negative coefficient for `position_tier_top_3` means pages in the top-three position category are associated with a lower predicted probability of decline relative to the reference category.

Other influential features include `impression_tier_low`, `impression_tier_excellent`, `word_count_tier_1000-2000`, and `log_clicks_90d`. These are model associations, not causal effects. The features are plausible because traffic visibility, search position, clicks, and content characteristics can distinguish different types of content performance, but the model does not establish that changing any one feature will cause a page to stop declining.

In [60]:
# Inspect model errors

error_check = test_df[
    ["content_id", "client_id", TARGET]
].copy()

error_check["predicted_probability"] = test_prob
error_check["predicted_label"] = test_pred

# False positives: model predicted decline, but actual label is 0
false_positives = error_check[
    (error_check[TARGET] == 0) &
    (error_check["predicted_label"] == 1)
].sort_values("predicted_probability", ascending=False)

# False negatives: model predicted no decline, but actual label is 1
false_negatives = error_check[
    (error_check[TARGET] == 1) &
    (error_check["predicted_label"] == 0)
].sort_values("predicted_probability", ascending=True)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\n=== 3 False Positives ===")
display(false_positives.head(3))

print("\n=== 3 False Negatives ===")
display(false_negatives.head(3))

False positives: 1394
False negatives: 1175

=== 3 False Positives ===


,content_id,client_id,is_declining_label,predicted_probability,predicted_label
26614,content_7be5f150dc65,client_f369cb89fc,0,0.945100,1
20736,content_41baf0722ad9,client_8527a891e2,0,0.932088,1
12869,content_5d5653c4eb4f,client_4e07408562,0,0.917577,1



=== 3 False Negatives ===


,content_id,client_id,is_declining_label,predicted_probability,predicted_label
8407,content_d1e915d03c28,client_4e07408562,1,0.050920,0
29158,content_e18144cbd19d,client_4e07408562,1,0.055430,0
17690,content_c268b1716236,client_e629fa6598,1,0.064807,0


### Three concrete errors

The model makes both false-positive and false-negative errors. A false positive is a page predicted as declining when the observed label is not declining. A false negative is a declining page that the model does not identify.

The three examples below are inspected as difficult cases rather than treated as evidence that the model is wrong for a specific causal reason. Possible explanations include overlapping traffic patterns, content characteristics, search-position differences, and information not represented in the available features.

1. **False positive:** The model assigned a high decline probability, but the observed label was 0. This indicates that the available features looked similar to declining pages even though the observed outcome was not declining.

2. **False positive:** The model again predicted decline but the observed label was 0. This shows that the model can over-prioritize pages with strong signals associated with decline.

3. **False negative:** The observed label was 1, but the model predicted 0. This indicates that some declining pages do not have a feature pattern that the Logistic Regression model can easily distinguish.

### Final conclusion

On the same client-held-out test split, Logistic Regression achieved higher Precision@20 than the ML-07 rule baseline (0.70 vs 0.65), while the baseline achieved higher Precision@50 (0.74 vs 0.72). Therefore, the learned model is not uniformly better than the baseline. It appears useful for prioritizing a smaller top-20 review queue, while the simpler rule remains competitive for a larger top-50 queue. The result is decision-support evidence rather than proof that the model will improve content performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.